<a href="https://colab.research.google.com/github/tellanad/AI-For-Beginners/blob/main/10_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import re

# Step 1: Our very first training examples
examples = [
    ("show all circuits",
     "SELECT circuit_id FROM circuits LIMIT 100;"),

    ("show failed circuits",
     "SELECT circuit_id FROM circuits WHERE status = 'failed' LIMIT 100;"),

    ("count active circuits",
     "SELECT COUNT(*) FROM circuits WHERE status = 'active';")
]

# Step 2: Split English into words
def tokenize(text):
    return re.findall(r"[a-z_]+", text.lower())

# Step 3: Create a vocabulary
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

words = sorted({
    word
    for question, sql in examples
    for word in tokenize(question)
})

vocab = special_tokens + words

# Step 4: Assign numerical IDs
word_to_id = {word: i for i, word in enumerate(vocab)}

# Step 5: Convert a sentence into IDs
question = "show failed circuits"

tokens = tokenize(question)
ids = (
    [word_to_id["<bos>"]]
    + [word_to_id.get(t, word_to_id["<unk>"]) for t in tokens]
    + [word_to_id["<eos>"]]
)

print("Vocabulary:", word_to_id)
print("Tokens:", tokens)
print("Token IDs:", ids)

In [ ]:

events = [
    {"agent_id": "billing", "status": "success", "latency_ms": 120},
    {"agent_id": "orders", "status": "failed", "latency_ms": 300},
    {"agent_id": "billing", "status": "failed", "latency_ms": 180},
    {"agent_id": "billing", "status": "success", "latency_ms": 100},
    {"agent_id": "orders", "status": "success", "latency_ms": 200}
]


In [ ]:
import json
from typing import List, Dict, Any

def aggregate_agent_events(events: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates events to compute total, success, failure, and average latency per agent.
    Optimized to run in O(N) time with O(1) extra space complexity per unique agent
    by avoiding storing individual latencies and statuses in lists.
    """
    # Use integer and float accumulators for optimal space efficiency
    stats = {}

    for event in events:
        agent = event["agent_id"]
        status = event["status"]
        latency = event["latency_ms"]

        if agent not in stats:
            stats[agent] = {
                "total": 0,
                "success": 0,
                "failure": 0,
                "total_latency": 0.0
            }

        stats[agent]["total"] += 1
        if status == "success":
            stats[agent]["success"] += 1
        elif status == "failed":
            stats[agent]["failure"] += 1

        stats[agent]["total_latency"] += latency

    # Generate the formatted response structure
    return {
        agent: {
            "total": data["total"],
            "success": data["success"],
            "failure": data["failure"],
            "avg_latency_ms": round(data["total_latency"] / data["total"], 2) if data["total"] > 0 else 0.0
        }
        for agent, data in stats.items()
    }

# --- Test Cases ---
def run_tests():
    test_events = [
        {"agent_id": "billing", "status": "success", "latency_ms": 120},
        {"agent_id": "orders", "status": "failed", "latency_ms": 300},
        {"agent_id": "billing", "status": "failed", "latency_ms": 180},
        {"agent_id": "billing", "status": "success", "latency_ms": 100},
        {"agent_id": "orders", "status": "success", "latency_ms": 200}
    ]

    result = aggregate_agent_events(test_events)

    # Assertions for validation
    assert result["billing"]["total"] == 3
    assert result["billing"]["success"] == 2
    assert result["billing"]["failure"] == 1
    assert result["billing"]["avg_latency_ms"] == 133.33

    assert result["orders"]["total"] == 2
    assert result["orders"]["success"] == 1
    assert result["orders"]["failure"] == 1
    assert result["orders"]["avg_latency_ms"] == 250.0

    print("All tests passed successfully!")
    return result

# Run tests and output formatted JSON
final_output = run_tests()
print("\nOutput result:")
print(json.dumps(final_output, indent=4))

{
    "billing": {
        "total": 3,
        "success": 2,
        "failure": 1,
        "avg_latency_ms": 133.33
    },
    "orders": {
        "total": 2,
        "success": 1,
        "failure": 1,
        "avg_latency_ms": 250.0
    }
}


In [ ]:
import json
from collections import defaultdict

# Automatically initializes missing keys with an empty list
grouped_data = defaultdict(list)
for event in events:
    # No "if agent not in grouped_data:" check needed!
    grouped_data[event["agent_id"]].append(event["latency_ms"])

# To print it cleanly as a standard JSON-like dictionary:
print(json.dumps(dict(grouped_data), indent=4))

{
    "billing": [
        120,
        180,
        100
    ],
    "orders": [
        300,
        200
    ]
}


In [ ]:
import json
from collections import defaultdict
from typing import List, Dict, Any

def process_agent_metrics(events: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Aggregates metrics per agent in O(N) time and O(1) extra space per agent.
    Avoids storing list arrays of latencies/statuses to maximize efficiency.
    """
    # Initialize with default structure representing counts and running sums
    grouped_data = defaultdict(lambda: {"total": 0, "success": 0, "failure": 0, "total_latency": 0.0})

    for event in events:
        agent = event["agent_id"]
        status = event["status"]
        latency = event["latency_ms"]

        metrics = grouped_data[agent]
        metrics["total"] += 1
        metrics["total_latency"] += latency

        if status == "success":
            metrics["success"] += 1
        elif status == "failed":
            metrics["failure"] += 1

    # Map into the final formatted layout
    return {
        agent: {
            "total": data["total"],
            "success": data["success"],
            "failure": data["failure"],
            "avg_latency_ms": round(data["total_latency"] / data["total"], 2) if data["total"] > 0 else 0.0
        }
        for agent, data in grouped_data.items()
    }

# --- Unit Tests ---
def run_validation_tests():
    test_cases = [
        {"agent_id": "billing", "status": "success", "latency_ms": 120},
        {"agent_id": "orders", "status": "failed", "latency_ms": 300},
        {"agent_id": "billing", "status": "failed", "latency_ms": 180},
        {"agent_id": "billing", "status": "success", "latency_ms": 100},
        {"agent_id": "orders", "status": "success", "latency_ms": 200}
    ]

    result = process_agent_metrics(test_cases)

    # Verification assertions
    assert result["billing"]["total"] == 3
    assert result["billing"]["success"] == 2
    assert result["billing"]["failure"] == 1
    assert result["billing"]["avg_latency_ms"] == 133.33

    assert result["orders"]["total"] == 2
    assert result["orders"]["success"] == 1
    assert result["orders"]["failure"] == 1
    assert result["orders"]["avg_latency_ms"] == 250.0

    print("All test assertions passed successfully!")
    return result

# Run the tests and print output
final_output = run_validation_tests()
print("\nFinal Output Result:")
print(json.dumps(final_output, indent=4))

All test assertions passed successfully!

Final Output Result:
{
    "billing": {
        "total": 3,
        "success": 2,
        "failure": 1,
        "avg_latency_ms": 133.33
    },
    "orders": {
        "total": 2,
        "success": 1,
        "failure": 1,
        "avg_latency_ms": 250.0
    }
}


Our first implementation will measure startup time, input tokens, generated tokens, generation latency, and average output-token throughput.

In [3]:


import time
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

# 1. Load the model and tokenizer
start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model = model.to("cpu")
model.eval()

load_time = time.perf_counter() - start
print(f"Model load time: {load_time:.2f}s")

# 2. Keep the model loaded between requests
while True:
    prompt = input("\nYou: ").strip()

    if prompt.lower() == "quit":
        break
    if not prompt:
        continue

    # 3. Convert messages into model inputs
    t0 = time.perf_counter()

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    t1 = time.perf_counter()

    # 4. Generate with gradients disabled
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    t2 = time.perf_counter()

    # 5. Extract only newly generated tokens
    input_count = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_count:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    generate_seconds = t2 - t1
    token_count = len(new_tokens)

    print(f"\nModel: {answer}")
    print(f"Input tokens: {input_count}")
    print(f"Output tokens: {token_count}")
    print(f"Tokenization: {t1 - t0:.3f}s")
    print(f"Generation: {generate_seconds:.3f}s")
    print(
        f"Average output tokens/sec: "
        f"{token_count / max(generate_seconds, 1e-9):.2f}"
    )

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model load time: 10.46s

Model: Transformers are a fundamental component in electrical engineering, particularly in the design of power transmission and distribution systems. They are used to convert alternating current (AC) to direct current (DC) and vice versa
Input tokens: 33
Output tokens: 40
Tokenization: 0.052s
Generation: 4.902s
Average output tokens/sec: 8.16

Model: Neural networks are a type of machine learning model that uses a combination of algorithms and neural networks to learn patterns and relationships in data. They are a fundamental component of artificial intelligence (AI) and have
Input tokens: 54
Output tokens: 40
Tokenization: 0.001s
Generation: 8.192s
Average output tokens/sec: 4.88

You: Our first implementation will measure startup time, input tokens, generated tokens, generation latency, and average output-token throughput.

Model: Here's a sample implementation of the `startup_time`, `input_tokens`, `generated_tokens`, `generation_latency`, and `average_outpu

KeyboardInterrupt: Interrupted by user

In [ ]:


import time

short_prompt = "What is a database index?"

long_prompt = (
    "A database index helps locate rows efficiently. " * 50
    + "What is a database index?"
)

for label, prompt in [
    ("SHORT", short_prompt),
    ("LONG", long_prompt),
]:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )

    # Repeat once to reduce first-run overhead
    for repeat in range(2):
        start = time.perf_counter()

        with torch.inference_mode():
            model.generate(
                **inputs,
                max_new_tokens=1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        elapsed = time.perf_counter() - start

    print(f"{label}:")
    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Generation time: {elapsed:.3f}s")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SHORT:
  Input tokens: 35
  Generation time: 3.146s
LONG:
  Input tokens: 435
  Generation time: 14.126s
